In [1]:
import os

In [2]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main\\research'

In [3]:
from pathlib import Path
import os

# Notebook lives in research/; config paths are relative to project root
_cwd = Path.cwd().resolve()
if _cwd.name == "research":
    os.chdir(_cwd.parent)

In [4]:
%pwd

'C:\\Users\\nisch\\Downloads\\Chicken-Disease-Classification-Projects-main\\Chicken-Disease-Classification-Projects-main'

In [5]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
import sys
import os
import unittest

# ensure 1.0.2 expects assertRaisesRegexp (removed in modern Python)
if not hasattr(unittest.TestCase, "assertRaisesRegexp"):
    unittest.TestCase.assertRaisesRegexp = unittest.TestCase.assertRaisesRegex

# cwd is project root; package is in ./src (not ../src after chdir)
sys.path.insert(0, os.path.abspath("src"))

from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml, create_directories


In [7]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])


    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config


In [8]:
import os
import urllib.request as request
import zipfile
from cnnClassifier import logger
from cnnClassifier.utils.common import get_size

In [9]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  


    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)



In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-03-28 22:53:05,076: INFO: common: yaml file: config\config.yaml loaded successfully]


[2026-03-28 22:53:05,078: INFO: common: yaml file: params.yaml loaded successfully]


[2026-03-28 22:53:05,078: INFO: common: created directory at: artifacts]


[2026-03-28 22:53:05,079: INFO: common: created directory at: artifacts/data_ingestion]


[2026-03-28 22:53:06,090: INFO: 1170291011: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 11616915
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "adf745abc03891fe493c3be264ec012691fe3fa21d861f35a27edbe6d86a76b1"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 7410:266F88:995B33:AD75CA:69C8A221
Accept-Ranges: bytes
Date: Sun, 29 Mar 2026 03:53:06 GMT
Via: 1.1 varnish
X-Served-By: cache-mci680070-MCI
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1774756386.425134,VS0,VE405
Vary: Authorization,Accept-Encoding
Access-Control-Allow-Origin: *
Cross-Origin-Resource-Policy: cross-origin
X-Fastly-Request-ID: 795cd864af9d896bed05e5d6c4c42d6c63dc91d2
Expires: Sun, 29 Mar 2026 03:58:06 GMT
Source-Age: 0

]
